# Step 2 — Initial Alignment (Anchor-Slice 2D Cross-Correlation)

Automatically finds seed landmarks that replace manual BigWarp placement.

**Algorithm** (`find_anchor_slice_alignment`):
1. Estimate HCR tissue bottom/top from robust centroid statistics (center-half XY, median of edge cells).
2. Apply ~180° rotation from calibration template to CZ centroids.
3. Scan a grid of `(cz_z, hcr_z)` anchor-slice pairs:
   - CZ: near pial surface (top 200 µm of rotated CZ stack)
   - HCR: within calibrated fraction [10%–40%] of tissue thickness from z_bot
4. For each pair: build 2D density images → one-sided FFT CC → derive XY translation.
5. Score by XY-MNN count (CZ cells within `1.5 × XY_THRESHOLD_UM` of HCR after applying CC shift).
6. Best `(cz_z, hcr_z, ty, tx)` → full rigid transform `t = [tz, ty, tx]`.
7. Extract seed landmarks using Z-restricted mutual NN.

**Outputs** (in `/scratch/{subject}_{date}_coreg_initial/`):
- `*_landmarks_initial.csv` — seed landmark pairs
- `*_rigid_initial.json` — rotation matrix + translation
- `fig_initial_alignment.png` — diagnostic scatter plot

**Next step**: `step_3_auto_matching.ipynb` — iterative TPS expansion from these seeds.

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
SUBJECT_ID   = "790322"          # e.g. "790322", "788406", ...
CZSTACK_DATE = "2025-07-10"      # date string used in the coreg directory name

# GFP+ spot-count threshold for selecting HCR cells
GFP_COUNT_THRESHOLD = 5

# Anchor-slice CC parameters
SIGMA_UM        = 15.0   # Gaussian sigma for density images (µm)
VOXEL_UM        = 5.0    # grid resolution for density images (µm)
CZ_SLAB_HALF_UM = 20.0   # half-width of CZ Z slab (µm)
CZ_Z_STEP_UM    = 15.0   # CZ anchor Z step (µm)
CZ_MAX_DEPTH_UM = 200.0  # only scan top 200 µm of CZ stack (pial surface end)
HCR_SLAB_HALF_UM = 20.0  # half-width of HCR Z slab (µm)
HCR_Z_STEP_UM   = 15.0   # HCR anchor Z step (µm)

# Matching thresholds
XY_THRESHOLD_UM = 22.0   # XY-MNN acceptance radius (µm)
Z_WINDOW_UM     = 150.0  # Z half-window for seed extraction (µm)

# Minimum seeds for a valid alignment
MIN_SEEDS = 20

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CODE_DIR = Path("/root/capsule/code")
DATA_DIR = Path("/root/capsule/data")
SCRATCH  = Path("/scratch")

sys.path.insert(0, str(CODE_DIR))

from coreg_data_loading import (
    load_czstack_centroids,
    load_hcr_centroids,
    load_hcr_scales,
    load_spot_counts,
)
from coreg_alignment import (
    CZ_RESOLUTION_UM,
    find_anchor_slice_alignment,
    euler_from_rotation_matrix,
)

In [ ]:
# ── Load calibration template ────────────────────────────────────────────────
template_path = DATA_DIR / "coreg_transform_template.json"
if template_path.exists():
    with open(template_path) as f:
        template = json.load(f)
    print("Calibration template loaded.")
    for key, val in template.items():
        print(f"  {key}: {val:.3f}" if isinstance(val, float) else f"  {key}: {val}")
else:
    template = {}
    print("WARNING: coreg_transform_template.json not found; using defaults.")

In [ ]:
# ── Locate subject data ─────────────────────────────────────────────────────
coreg_dirs = sorted(DATA_DIR.glob(f"{SUBJECT_ID}_{CZSTACK_DATE}_ctl-czstack-hcr-coreg_*"))
if not coreg_dirs:
    raise FileNotFoundError(
        f"No coreg directory found for {SUBJECT_ID} / {CZSTACK_DATE}. "
        "Run step_1_process_files.ipynb first or check SUBJECT_ID / CZSTACK_DATE."
    )
coreg_dir = coreg_dirs[-1]
print(f"Coreg directory: {coreg_dir.name}")

# Load filepaths JSON produced by step_1; prefer *_filepaths_iter.json (has spot_488_counts_path)
fp_iter = sorted(coreg_dir.glob(f"{SUBJECT_ID}_{CZSTACK_DATE}_filepaths_iter.json"))
fp_base = sorted(coreg_dir.glob(f"{SUBJECT_ID}_{CZSTACK_DATE}_filepaths.json"))
fp_json_path = (fp_iter or fp_base or [])[0] if (fp_iter or fp_base) else None
if fp_json_path is None:
    raise FileNotFoundError(f"No filepaths JSON in {coreg_dir}")
with open(fp_json_path) as f:
    filepaths = json.load(f)
print(f"Filepaths JSON: {fp_json_path.name}")
print(json.dumps(filepaths, indent=2))

In [ ]:
# ── Load CZ centroids ────────────────────────────────────────────────────────
czstack_centroid_path = Path(filepaths["czstack_centroid_path"])
if not czstack_centroid_path.exists():
    # Fallback: file may have been saved to coreg_dir rather than scratch
    fallback = coreg_dir / f"{SUBJECT_ID}_{CZSTACK_DATE}_czstack_cell_centroids.csv"
    if fallback.exists():
        czstack_centroid_path = fallback
        print(f"Note: using centroid path in coreg_dir: {fallback.name}")
    else:
        raise FileNotFoundError(
            f"CZ centroid CSV not found at {czstack_centroid_path}\n"
            f"Fallback {fallback} also missing. Run step_1_process_files.ipynb."
        )
czstack_df = load_czstack_centroids(czstack_centroid_path)
print(f"CZ centroids: {len(czstack_df)} cells")
print(czstack_df.head(3))

In [ ]:
# ── Load HCR centroids (all cells + GFP+ subset) ────────────────────────────
hcr_scales = load_hcr_scales(filepaths["fused_json_file"])
print(f"HCR scales (µm/px): z={hcr_scales['scale_z']:.4f}, "
      f"y={hcr_scales['scale_y']:.4f}, x={hcr_scales['scale_x']:.4f}")
sc = np.array([hcr_scales["scale_z"], hcr_scales["scale_y"], hcr_scales["scale_x"]])

# All HCR centroids (for tissue surface estimation)
hcr_all_npy = filepaths.get("hcr_centroid_path")
if hcr_all_npy is None:
    raise FileNotFoundError("hcr_centroid_path not found in filepaths JSON")
hcr_full_df = load_hcr_centroids(hcr_all_npy)
hcr_all_um = np.stack([
    hcr_full_df["hcr_z"].values * sc[0],
    hcr_full_df["hcr_y"].values * sc[1],
    hcr_full_df["hcr_x"].values * sc[2],
], axis=1)
print(f"HCR all cells: {len(hcr_full_df)}")
print(f"HCR volume (µm): z=[{hcr_all_um[:,0].min():.0f}, {hcr_all_um[:,0].max():.0f}], "
      f"y=[{hcr_all_um[:,1].min():.0f}, {hcr_all_um[:,1].max():.0f}], "
      f"x=[{hcr_all_um[:,2].min():.0f}, {hcr_all_um[:,2].max():.0f}]")

In [ ]:
# ── Load spot counts and build GFP+ HCR subset ──────────────────────────────
# load_spot_counts searches:
#   1. fallback_coreg_dir / "{subject}_{date}_spot_488_counts.csv"
#   2. hcr_processed_dir / image_spot_detection / channel_488_spots / spots.csv
hcr_processed_dir = Path(filepaths["hcr_centroid_path"]).parent.parent
spot_counts = load_spot_counts(
    hcr_processed_dir=hcr_processed_dir,
    hcr_metrics_df=hcr_full_df,
    fallback_coreg_dir=coreg_dir,
    subject_id=SUBJECT_ID,
    czstack_date=CZSTACK_DATE,
)
print(f"Spot counts loaded: {len(spot_counts)} cells, columns: {list(spot_counts.columns)}")

hcr_with_spots = hcr_full_df.merge(
    spot_counts[["hcr_cell_id", "counts"]],
    on="hcr_cell_id", how="left",
).fillna({"counts": 0})

hcr_gfp_df = hcr_with_spots[hcr_with_spots["counts"] >= GFP_COUNT_THRESHOLD].reset_index(drop=True)
print(f"GFP+ HCR cells (count ≥ {GFP_COUNT_THRESHOLD}): {len(hcr_gfp_df)} / {len(hcr_full_df)}")

In [ ]:
# ── Run anchor-slice alignment ────────────────────────────────────────────────
seed_df, R_best, t_best, score, info = find_anchor_slice_alignment(
    czstack_df=czstack_df,
    hcr_gfp_df=hcr_gfp_df,
    hcr_all_um=hcr_all_um,
    hcr_scales=hcr_scales,
    template=template,
    czstack_res_um=CZ_RESOLUTION_UM,
    cz_slab_half_um=CZ_SLAB_HALF_UM,
    cz_z_step_um=CZ_Z_STEP_UM,
    cz_anchor_max_depth_um=CZ_MAX_DEPTH_UM,
    hcr_slab_half_um=HCR_SLAB_HALF_UM,
    hcr_z_step_um=HCR_Z_STEP_UM,
    sigma_um=SIGMA_UM,
    voxel_um=VOXEL_UM,
    xy_threshold_um=XY_THRESHOLD_UM,
    z_window_um=Z_WINDOW_UM,
    verbose=True,
)

print(f"\nResult: {len(seed_df)} seed landmarks, full-volume XY-MNN = {score}/{len(czstack_df)}")
print(f"Rigid t (tz, ty, tx): [{t_best[0]:.0f}, {t_best[1]:.0f}, {t_best[2]:.0f}] µm")
print(f"Anchor info: {info}")

assert len(seed_df) >= MIN_SEEDS, (
    f"Only {len(seed_df)} seeds found (need ≥ {MIN_SEEDS}). Try:\n"
    "  - Increasing Z_WINDOW_UM\n"
    "  - Lowering GFP_COUNT_THRESHOLD\n"
    "  - Checking that filepaths.json points to the correct HCR data"
)

In [ ]:
# ── Diagnostics: 3-panel scatter plot ────────────────────────────────────────
cz_res = np.asarray(CZ_RESOLUTION_UM)
cz_um_all = np.stack([
    czstack_df["czstack_z"].values * cz_res[0],
    czstack_df["czstack_y"].values * cz_res[1],
    czstack_df["czstack_x"].values * cz_res[2],
], axis=1)
cz_aligned_um = (R_best @ cz_um_all.T).T + t_best

hcr_gfp_um = np.stack([
    hcr_gfp_df["hcr_z"].values * sc[0],
    hcr_gfp_df["hcr_y"].values * sc[1],
    hcr_gfp_df["hcr_x"].values * sc[2],
], axis=1)

tz_r, tx_r, ty_r = euler_from_rotation_matrix(R_best)
quality = len(seed_df) / len(czstack_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f"{SUBJECT_ID} — Initial alignment  "
    f"seeds={len(seed_df)}/{len(czstack_df)} ({100*quality:.0f}%)  "
    f"XY-MNN={score}\n"
    f"rotation: tz={tz_r:.1f}° tx={tx_r:.1f}° ty={ty_r:.1f}°  "
    f"t=[{t_best[0]:.0f}, {t_best[1]:.0f}, {t_best[2]:.0f}] µm",
    fontsize=11,
)

views = [(1, 2, "YX (top view)"), (0, 2, "ZX (side view)"), (0, 1, "ZY (front view)")]
for ax, (dim_a, dim_b, title) in zip(axes, views):
    ax.scatter(
        hcr_gfp_um[:, dim_b], hcr_gfp_um[:, dim_a],
        s=3, c="steelblue", alpha=0.3, label="HCR GFP+",
    )
    ax.scatter(
        cz_aligned_um[:, dim_b], cz_aligned_um[:, dim_a],
        s=8, c="tomato", alpha=0.6, label="CZ aligned",
    )
    # Draw seed match lines
    for _, row in seed_df.iterrows():
        cz_pt = np.array([
            row["czstack_z"] * cz_res[0],
            row["czstack_y"] * cz_res[1],
            row["czstack_x"] * cz_res[2],
        ])
        cz_pt_aligned = (R_best @ cz_pt) + t_best
        hcr_pt = np.array([
            row["hcr_z"] * sc[0],
            row["hcr_y"] * sc[1],
            row["hcr_x"] * sc[2],
        ])
        ax.plot(
            [cz_pt_aligned[dim_b], hcr_pt[dim_b]],
            [cz_pt_aligned[dim_a], hcr_pt[dim_a]],
            "-", c="gold", lw=0.7, alpha=0.5,
        )
    ax.set_xlabel(["z", "y", "x"][dim_b] + " (µm)")
    ax.set_ylabel(["z", "y", "x"][dim_a] + " (µm)")
    ax.set_title(title)
    ax.legend(markerscale=2, fontsize=8)

plt.tight_layout()

out_dir = SCRATCH / f"{SUBJECT_ID}_{CZSTACK_DATE}_coreg_initial"
out_dir.mkdir(parents=True, exist_ok=True)
fig_path = out_dir / "fig_initial_alignment.png"
fig.savefig(fig_path, dpi=150)
print(f"Figure saved: {fig_path}")
plt.show()

In [ ]:
# ── Z offset diagnostics ─────────────────────────────────────────────────────
# The rigid tz is just the anchor-slice offset; TPS in step_3 handles
# the full spatially-variable non-rigid Z compression (~2.5–3.5×).
z_offsets = []
for _, row in seed_df.iterrows():
    cz_pt_aligned = (R_best @ np.array([
        row["czstack_z"] * cz_res[0],
        row["czstack_y"] * cz_res[1],
        row["czstack_x"] * cz_res[2],
    ])) + t_best
    z_offsets.append(float(cz_pt_aligned[0]) - row["hcr_z"] * sc[0])

z_offsets = np.array(z_offsets)
print(f"Seed Z residual (CZ_aligned_z − HCR_matched_z) in µm:")
print(f"  median = {np.median(z_offsets):+.1f}")
print(f"  std    = {np.std(z_offsets):.1f}")
print(f"  range  = [{z_offsets.min():.1f}, {z_offsets.max():.1f}]")
print()
print("A non-zero Z residual is expected — TPS in step_3 models spatially-variable Z compression.")

In [ ]:
# ── Save seed landmarks ───────────────────────────────────────────────────────
out_dir = SCRATCH / f"{SUBJECT_ID}_{CZSTACK_DATE}_coreg_initial"
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / f"{SUBJECT_ID}_{CZSTACK_DATE}_landmarks_initial.csv"
seed_df.to_csv(out_csv, index=False)
print(f"Saved {len(seed_df)} seed landmarks → {out_csv}")
print(seed_df.head(10).to_string(index=False))

In [ ]:
# ── Save rigid transform ──────────────────────────────────────────────────────
rigid_info = {
    "subject_id":    SUBJECT_ID,
    "czstack_date":  CZSTACK_DATE,
    "xy_mnn_score":  int(score),
    "n_seeds":       len(seed_df),
    "cz_anchor_um":  info["cz_anchor_um"],
    "hcr_anchor_um": info["hcr_anchor_um"],
    "z_bot_um":      info["z_bot"],
    "z_top_um":      info["z_top"],
    "R":             R_best.tolist(),
    "t":             t_best.tolist(),
}
rigid_json = out_dir / f"{SUBJECT_ID}_{CZSTACK_DATE}_rigid_initial.json"
with open(rigid_json, "w") as f:
    json.dump(rigid_info, f, indent=2)
print(f"Rigid transform saved → {rigid_json}")
print(json.dumps(rigid_info, indent=2))